# Task 1

## 1. Espacio de estados S

La acción anterior no importa, lo que importa es dónde está el dron ahora, porque podemos definir estados por celda dentro de la "ciuddad", como respuesta general pero con otros parámetros para darle contexto.

Pero para los drones no alcanza usar solo una celda. Por ejemplo, si el dron está en `(3, 4)`, no puede decidir bien si no sabe si todavía lleva el paquete, si ya lo entregó o cuánta batería tiene. Dos drones en la misma casilla podrían tener que hacer cosas totalmente distintas.

Entonces proponemos este estado:

`s = (x, y, b, p, d)`

- `(x, y)`: posición del dron en la cuadrícula. Se necesita para saber qué movimientos son posibles y cuánto falta para llegar.
- `b`: batería restante, por ejemplo de 0 a 10. Se necesita porque determina si el dron puede completar el viaje y volver a la base.
- `p`: estado del paquete: `1` si el dron todavía lo lleva y `0` si ya lo entregó. Antes de entregar debe ir al destino; después debe regresar.
- `d`: coordenada del destino asignado. Es necesaria si los pedidos pueden ir a distintas casillas.

La base es siempre la casilla central `(2,2)`, por eso no hace falta guardarla dentro de todos los estados. También suponemos que los edificios o zonas prohibidas son fijos y conocidos: pueden estar definidos una sola vez en el mapa. Omitimos cosas como el nombre del cliente, el color del dron o el número del pedido porque no cambian el resultado del siguiente movimiento.

Estamos omitiendo viento, tráfico de otros drones y estado real de la batería. Es una simplificación. La consecuencia es que el MDP no representa toda la realidad: dos estados que se ven iguales podrían tener riesgos distintos. Aun así, sirve como primer modelo para saber si la idea funciona antes de hacer algo mucho más complejo.

La propiedad de Markov se cumple razonablemente porque, al conocer posición, batería, paquete y destino, podemos decidir la siguiente acción y estimar qué pasará sin necesitar la acción anterior.

## 2. Espacio de acciones A

Las acciones son: `norte`, `sur`, `este`, `oeste`, `entregar` y `esperar`. Son acciones discretas porque el dron escoge una opción de una lista; no puede escoger cualquier dirección decimal, como 37.4 grados.

Hay acciones que se deben restringir:

- No puede ir al norte si ya está en la fila superior, ni salir por ningún borde del tablero.
- No puede ir hacia una casilla con edificio o zona prohibida.
- Solo puede usar `entregar` si está exactamente en el destino y aún tiene el paquete.

Esto se modela con `A(s)`, que significa “las acciones permitidas en el estado s”. Por ejemplo, en una esquina solo habrá dos movimientos válidos. También podríamos dejar que el dron elija una acción inválida, pero entonces lo dejamos en la misma casilla y le damos un castigo; para el laboratorio es más claro simplemente no incluirla en `A(s)`.

## 3. Función de recompensa r(s,a,s')

La recompensa es el puntaje que guía al dron. La empresa quiere entregar rápido, gastar poca batería y no tener accidentes. Como estas metas compiten entre sí, usamos estas recompensas:

- Cada movimiento válido: `-1`. Así evita dar vueltas y hace entregas eficientes.
- Cada unidad de batería usada: `-0.5`. Así prefiere rutas que consuman menos.
- Entregar el paquete: `+30`.
- Volver a la base después de entregar: `+40`.
- Chocar, entrar a una zona prohibida o quedarse sin batería: `-100` y termina la misión.

El castigo de seguridad es el más grande porque un accidente cuesta mucho más que tardar un paso extra. Si el premio por entregar fuera demasiado alto, el agente podría tomar rutas peligrosas. Si el costo de batería fuera demasiado alto, podría evitar hacer entregas largas aunque sean necesarias. Si el castigo por accidente fuera pequeño, podría aprender a arriesgarse demasiado.

## 4. Función de transición p(s' | s,a)

En este ejemplo de tablero básico las transiciones podrían ser deterministas: si voy al este, siempre avanzo una casilla. Para una ciudad real es mejor usar una transición estocástica, o sea, con probabilidades, porque hay viento, errores de GPS, obstáculos temporales o fallos de navegación.

Ejemplos:

1. Si el dron está en `(1,1,7,1,d)` y usa `este`, tiene probabilidad `0.90` de llegar a `(2,1,6,1,d)` y probabilidad `0.10` de quedarse en `(1,1,6,1,d)` por una ráfaga de viento. Casi siempre el vuelo funciona, pero aun cuando no avanza gasta batería tratando de hacerlo, y sería de considerar el viento para el gasto de batería.
2. Si está en `(3,3,5,1,d)` y usa `norte`, tiene probabilidad `0.85` de llegar a `(3,2,4,1,d)`, probabilidad `0.10` de quedarse donde está y probabilidad `0.05` de caer o tener un accidente. La probabilidad pequeña de accidente representa una condición urbana peligrosa.
3. Si está en el destino `(4,4,4,1,(4,4))` y usa `entregar`, cambia a `(4,4,4,0,(4,4))` con probabilidad `0.99`; con probabilidad `0.01` la entrega no se confirma y permanece con el paquete. Es normal que entregar sea muy confiable, pero no perfecto, puede haber alguien que no reciba, o incluso problemas con el paquete en el momento de entrega.

Estos números son supuestos iniciales. Una empresa real debería estimarlos con datos de vuelos y accidentes.

## 5. Factor de descuento gamma

Proponemos `γ = 0.95`. Gamma. En este caso queremos que la entrega y el regreso sean importantes, pero que el dron también prefiera terminarlos pronto. Un valor de 0.95 hace que una ruta segura un poco más larga siga valiendo la pena, sin hacer que el agente pierda demasiado tiempo dando vueltas.


# Task 2

## 1. Casos donde se viola Markov

**Caso 1: el viento.** Dos drones pueden estar en la misma posición, con la misma batería y paquete, pero uno puede tener viento fuerte y el otro viento normal. Si no guardamos el viento, la probabilidad de avanzar o fallar no es igual en ambos casos. Para restaurar Markov agregamos una variable `w` al estado: `w = normal, suave o fuerte`. El costo es que el número de estados se multiplica por tres. Si además guardamos dirección y velocidad del viento, el modelo crece aún más.

**Caso 2: batería desgastada.** Dos drones con batería nivel 5 no necesariamente gastan igual: uno puede tener una batería nueva y otro una dañada. Sin incluir esa condición, el siguiente estado depende de la historia de uso de la batería. Podemos agregar `h`, salud de batería: `buena, media o mala`. El costo vuelve a multiplicar los estados y necesitamos estimar esa salud con sensores.

También podría pasar con otros drones: para saber el riesgo de choque habría que agregar las posiciones y tal vez direcciones de los drones cercanos. Eso haría crecer el espacio de estados muy rápido, porque cada dron adicional puede estar en muchas casillas.

## 2. ¿El dron observa todo el estado?

No es muy razonable asumir que ve el estado completo. El GPS puede tener error; el dron no conoce perfectamente el viento entre edificios; la salud real de la batería es una estimación; y puede no detectar a tiempo aves, cables, obstáculos o drones que estén ocultos.

Un MDP supone que el agente conoce el estado real, por ejemplo que sabe con certeza que está en `(2,3)` y que el viento es suave. Eso hace que el modelo sea más fácil de programar y es aceptable para este laboratorio. En la realidad sería más apropiado un POMDP, que es un MDP parcialmente observable. En un POMDP el dron no ve todo: recibe observaciones imperfectas de GPS, cámara y sensores, y decide usando una estimación de la situación. Es más realista, pero más difícil de resolver.

## 3. ¿Tarea episódica o continua?

Para este problema la modelaría como episódica. Un episodio inicia cuando el dron sale de la base con un paquete y termina cuando lo entrega y regresa a la base, o cuando ocurre un accidente o se queda sin batería. Cada pedido es una misión completa y separada.

Con una tarea episódica tiene sentido dar premios grandes al entregar y al regresar, y usar `γ = 0.95`: queremos completar esta misión pronto y de manera segura.

Si se modelara como continua, el dron nunca termina: al volver toma otro paquete y sigue trabajando. Entonces la recompensa debería incluir beneficios repetidos por entregas y costos constantes por tiempo, batería y riesgo; no dependería solo de un premio final. También convendría un gamma más cercano a 1, por ejemplo `0.99`, porque importan mucho las entregas futuras de toda la jornada.
